# SDT 1,500-record DPO pilot

Single-judge pilot with LFM2.5-1.2B-Instruct. Test data stays locked until validation selection is frozen.

## 1. GPU, branch, and installation

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU."
subprocess.run(["nvidia-smi"], check=True)

REPO_URL = "https://github.com/Rana-Ezzeddine/SDT.git"
BRANCH = "1500-record-dpo-pipeline"
REPO_DIR = Path("/content/SDT")
if REPO_DIR.exists():
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "switch", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())

## 2. Upload the raw JSON

In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Upload only sdt_results_1500.json."
RAW_DATA = Path("data/raw/sdt_results_1500.json")
RAW_DATA.parent.mkdir(parents=True, exist_ok=True)
RAW_DATA.write_bytes(next(iter(uploaded.values())))
print("Saved", RAW_DATA, "bytes=", RAW_DATA.stat().st_size)

## 3. Build provisional pairs and inspect the audit

In [ ]:
import json

PAIRS = Path("data/processed/dpo_pairs_1500.jsonl")
PAIR_REPORT = Path("data/processed/pair_report_1500.json")
subprocess.run([
    "sdt-build-pairs", "--input", str(RAW_DATA), "--output", str(PAIRS),
    "--report", str(PAIR_REPORT), "--label-mode", "single_judge_pilot",
    "--use-supplied-aggregates", "--min-common-judges", "1",
    "--min-margin", "0.10", "--train-share", "0.80",
    "--validation-share", "0.10", "--seed", "42",
], check=True)
report = json.loads(PAIR_REPORT.read_text())
for key in ["records", "possible_pairs", "retained_pairs", "retained_prompts", "retained_by_split", "retained_by_comparison_type", "exclusion_reasons"]:
    print(f"{key}: {report.get(key)}")
print("judgment_evidence:", json.dumps(report["judgment_evidence"], indent=2))

## 4. Run tests and inspect LFM token lengths

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)

import numpy as np
from transformers import AutoTokenizer

BASELINE_MODEL = "LiquidAI/LFM2.5-1.2B-Instruct"
MAX_LENGTH = 2048
tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL)
retained = [json.loads(line) for line in PAIRS.read_text().splitlines() if json.loads(line)["retain"]]
lengths = []
for row in retained:
    for response in (row["chosen"], row["rejected"]):
        messages = [{"role": "user", "content": row["prompt"]}, {"role": "assistant", "content": response}]
        lengths.append(len(tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)))
print({"median": int(np.median(lengths)), "p95": int(np.percentile(lengths, 95)), "p99": int(np.percentile(lengths, 99)), "max": max(lengths), "over_2048": sum(x > MAX_LENGTH for x in lengths)})

## 5. Optional validation-only tuning, then freeze one checkpoint

In [ ]:
RUN_TUNING = False
EXPERIMENTS = [
    {"name": "lr5e7_beta010", "learning_rate": 5e-7, "beta": 0.10, "num_train_epochs": 1},
    {"name": "lr1e6_beta010", "learning_rate": 1e-6, "beta": 0.10, "num_train_epochs": 1},
    {"name": "lr5e7_beta020", "learning_rate": 5e-7, "beta": 0.20, "num_train_epochs": 1},
]
if not RUN_TUNING:
    EXPERIMENTS = EXPERIMENTS[:1]
print("Candidates:", EXPERIMENTS)

In [ ]:
import copy, hashlib, yaml
import pandas as pd

fingerprint = hashlib.sha256(RAW_DATA.read_bytes()).hexdigest()[:12]
RUN_ROOT = Path("outputs") / f"pilot-1500-{fingerprint}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
base_config = yaml.safe_load(Path("configs/pilot_1500_lfm.yaml").read_text())
base_config.update({"pairs_file": str(PAIRS.resolve()), "max_length": MAX_LENGTH})

baseline_val = RUN_ROOT / "baseline-validation.json"
baseline_val_details = RUN_ROOT / "baseline-validation-pairs.jsonl"
subprocess.run(["sdt-evaluate-pairs", "--pairs", str(PAIRS), "--model", BASELINE_MODEL,
                "--split", "validation", "--max-length", str(MAX_LENGTH), "--all-confidence",
                "--output", str(baseline_val), "--details", str(baseline_val_details)], check=True)

results = []
for experiment in EXPERIMENTS:
    cfg = copy.deepcopy(base_config)
    cfg.update({k: v for k, v in experiment.items() if k != "name"})
    model_dir = RUN_ROOT / experiment["name"] / "model"
    cfg["output_dir"] = str(model_dir.resolve())
    config_path = Path("configs") / f"notebook_{experiment['name']}.yaml"
    config_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    if not (model_dir / "config.json").exists():
        subprocess.run(["sdt-train-dpo", "--config", str(config_path)], check=True)
    candidate_details = model_dir.parent / "validation-pairs.jsonl"
    subprocess.run(["sdt-evaluate-pairs", "--pairs", str(PAIRS), "--model", str(model_dir),
                    "--split", "validation", "--max-length", str(MAX_LENGTH), "--all-confidence",
                    "--output", str(model_dir.parent / "validation-summary.json"), "--details", str(candidate_details)], check=True)
    comparison_path = model_dir.parent / "validation-comparison.json"
    subprocess.run(["sdt-compare-evaluations", "--baseline-details", str(baseline_val_details),
                    "--dpo-details", str(candidate_details), "--output", str(comparison_path),
                    "--beta", str(cfg["beta"])], check=True)
    comparison = json.loads(comparison_path.read_text())
    training_metrics = json.loads((model_dir / "validation_metrics.json").read_text())
    primary = comparison["primary_dpo_relative_metrics"]
    results.append({**experiment, "model": str(model_dir),
                    "prompt_macro_implicit_accuracy": primary["implicit_reward_prompt_macro_accuracy"],
                    "mean_implicit_margin": primary["mean_implicit_reward_margin"],
                    "validation_loss": training_metrics.get("validation_loss")})

table = pd.DataFrame(results).sort_values(["prompt_macro_implicit_accuracy", "mean_implicit_margin", "validation_loss"], ascending=[False, False, True]).reset_index(drop=True)
display(table)
best = table.iloc[0].to_dict()
DPO_MODEL = best["model"]
SELECTED_BETA = float(best["beta"])
(RUN_ROOT / "selected-configuration.json").write_text(json.dumps(best, indent=2, default=str))
print("Frozen checkpoint:", DPO_MODEL)

## 6. Verify the checkpoint

In [ ]:
subprocess.run(["sdt-verify-checkpoint", "--baseline-model", BASELINE_MODEL,
                "--trained-model", DPO_MODEL, "--output", str(RUN_ROOT / "checkpoint-change.json")], check=True)
print(json.loads((RUN_ROOT / "checkpoint-change.json").read_text()))

## 7. Locked fixed-pair test (enable only after freezing the checkpoint)

In [ ]:
RUN_LOCKED_TEST = False
TEST_DIR = RUN_ROOT / "test"
if not RUN_LOCKED_TEST:
    print("Test remains locked. Set RUN_LOCKED_TEST=True only after validation selection.")
else:
    TEST_DIR.mkdir(parents=True, exist_ok=True)
    for name, model in [("baseline", BASELINE_MODEL), ("dpo", DPO_MODEL)]:
        subprocess.run(["sdt-evaluate-pairs", "--pairs", str(PAIRS), "--model", model,
                        "--split", "test", "--max-length", str(MAX_LENGTH), "--all-confidence",
                        "--output", str(TEST_DIR / f"{name}-summary.json"),
                        "--details", str(TEST_DIR / f"{name}-pairs.jsonl")], check=True)
    subprocess.run(["sdt-compare-evaluations", "--baseline-details", str(TEST_DIR / "baseline-pairs.jsonl"),
                    "--dpo-details", str(TEST_DIR / "dpo-pairs.jsonl"),
                    "--output", str(TEST_DIR / "fixed-pair-comparison.json"),
                    "--details-output", str(TEST_DIR / "relative-pairs.jsonl"),
                    "--beta", str(SELECTED_BETA)], check=True)
    fixed = json.loads((TEST_DIR / "fixed-pair-comparison.json").read_text())
    print(json.dumps(fixed["primary_dpo_relative_metrics"], indent=2))
    print(json.dumps(fixed["secondary_absolute_likelihood_metrics"], indent=2))

## 8. Fresh baseline and DPO generations on the same test prompts

In [ ]:
if not RUN_LOCKED_TEST:
    print("Skipped because the test is locked.")
else:
    for name, model in [("baseline", BASELINE_MODEL), ("dpo", DPO_MODEL)]:
        subprocess.run(["sdt-generate-responses", "--pairs", str(PAIRS), "--model", model,
                        "--output", str(TEST_DIR / f"{name}-generations.jsonl"),
                        "--split", "test", "--max-input-length", str(MAX_LENGTH),
                        "--max-new-tokens", "512", "--temperature", "0", "--seed", "42"], check=True)

## 9. Optional blinded LLM-as-judge

Use an independent judge. Add JUDGE_API_KEY to Colab secrets and fill the model and compatible endpoint.

In [ ]:
RUN_LLM_JUDGE = False
JUDGE_MODEL = ""
JUDGE_API_URL = "https://api.openai.com/v1/chat/completions"

if RUN_LLM_JUDGE:
    from google.colab import userdata
    assert RUN_LOCKED_TEST and JUDGE_MODEL, "Generate the locked-test answers and set JUDGE_MODEL."
    os.environ["JUDGE_API_KEY"] = userdata.get("JUDGE_API_KEY")
    subprocess.run(["sdt-judge-generations", "--baseline", str(TEST_DIR / "baseline-generations.jsonl"),
                    "--dpo", str(TEST_DIR / "dpo-generations.jsonl"),
                    "--details", str(TEST_DIR / "generation-judgments.jsonl"),
                    "--summary", str(TEST_DIR / "generation-judge-summary.json"),
                    "--judge-model", JUDGE_MODEL, "--api-url", JUDGE_API_URL], check=True)
    print((TEST_DIR / "generation-judge-summary.json").read_text())
else:
    print("LLM judge disabled. Fixed-pair evaluation and generation can still be completed first.")

## 10. Optional Drive backup

In [ ]:
BACKUP_TO_DRIVE = False
if BACKUP_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/SDT_DPO_Pilot") / RUN_ROOT.name
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(RUN_ROOT, destination)
    print("Saved to", destination)